In [44]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print(api_key is not None)

True


In [45]:
from pathlib import Path
from urllib.request import urlretrieve

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.memory import ConversationBufferMemory

In [46]:
files_dir = Path("./files")
files_dir.mkdir(exist_ok=True)

document_url = "https://gist.githubusercontent.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223/raw/document.txt"
document_path = files_dir / "document.txt"

if not document_path.exists():
    urlretrieve(document_url, document_path)

print(document_path.exists())

True


In [47]:
loader = TextLoader(document_path, encoding="utf-8")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator ="\n",
    chunk_size = 600,
    chunk_overlap=100,
)

docs= loader.load_and_split(text_splitter=splitter)

print(len(docs))
print(docs[0].page_content[:500])

34
Chapter 3
'There are three stages in your reintegration,' said O'Brien. 'There is
learning, there is understanding, and there is acceptance. It is time for
you to enter upon the second stage.'
As always, Winston was lying flat on his back. But of late his bonds were
looser. They still held him to the bed, but he could move his knees a
little and could turn his head from side to side and raise his arms from
the elbow. The dial, also, had grown to be less of a terror. He could
evade its pangs if h


In [48]:
cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    cache_dir,
)

vectorstore = FAISS.from_documents(
    docs,
    cached_embeddings,
)

retriever = vectorstore.as_retriever(
    # search_kwargs={"k": len(docs)}
    search_kwargs={"k": 12}
)

print("Vector store is ready")

Vector store is ready


In [49]:
# test_docs = retriever.invoke("Is Aaronson guilty?")
# print(len(test_docs),"개","\n")

# for index, doc in enumerate(test_docs):
#     print("=" * 80)
#     print(f"Document {index + 1}")
#     print(doc.page_content[:1000])

test_docs = retriever.invoke("Is Aaronson guilty?")

print(len(test_docs), "documents retrieved")

for index, doc in enumerate(test_docs):
    content = doc.page_content
    if "Aaronson" in content or "Jones" in content or "Rutherford" in content:
        print("=" * 80)
        print(f"Document {index + 1}")
        print(content)

12 documents retrieved
Document 3
Yes, even... He could not fight against the Party any longer. Besides,
the Party was in the right. It must be so; how could the immortal,
collective brain be mistaken? By what external standard could you check
its judgements? Sanity was statistical. It was merely a question of
learning to think as they thought. Only----!
The pencil felt thick and awkward in his fingers. He began to write down
the thoughts that came into his head. He wrote first in large clumsy
capitals:
FREEDOM IS SLAVERY
Then almost without a pause he wrote beneath it:
TWO AND TWO MAKE FIVE
But then there came a sort of check. His mind, as though shying away from
something, seemed unable to concentrate. He knew that he knew what came
next, but for the moment he could not recall it. When he did recall it,
it was only by consciously reasoning out what it must be: it did not come
of its own accord. He wrote:
GOD IS POWER
He accepted everything. The past was alterable. The past never had 

In [50]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.1,
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="question",
    output_key="answer",
    return_messages=True,
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def load_memory(_):
    return memory.load_memory_variables({})["chat_history"]

In [51]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful assistant.

Answer the user's question using only the context below.
If the answer is not in the context, say you don't know.
Do not make up an answer.

Context:
{context}
""",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ]
)

In [52]:
chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
        "chat_history": RunnableLambda(load_memory),
    }
    | prompt
    | llm
)

In [53]:
def invoke_chain(question):
    response = chain.invoke(question)
    
    memory.save_context(
        {"question": question},
        {"answer": response.content},
    )
    
    return response

In [54]:
questions = [
    "Is Aaronson guilty?",
    "What message did he write in the table?",
    "Who is Julia?",
]

for question in questions:
    response = invoke_chain(question)
    print("Question:", question)
    print("Answer:", response.content)
    print("-" * 80)

Question: Is Aaronson guilty?
Answer: Yes, Aaronson, along with Jones and Rutherford, were guilty of the crimes they were charged with.
--------------------------------------------------------------------------------
Question: What message did he write in the table?
Answer: Almost unconsciously, he traced with his finger in the dust on the table: 2+2=5.
--------------------------------------------------------------------------------
Question: Who is Julia?
Answer: Julia is a character who was involved with Winston in the novel.
--------------------------------------------------------------------------------


In [55]:
memory.load_memory_variables({})

{'chat_history': [HumanMessage(content='Is Aaronson guilty?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Yes, Aaronson, along with Jones and Rutherford, were guilty of the crimes they were charged with.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What message did he write in the table?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Almost unconsciously, he traced with his finger in the dust on the table: 2+2=5.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Who is Julia?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Julia is a character who was involved with Winston in the novel.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}